# 2.3 Refactorización de P2 para la etapa 2

Lo explorado en `1.3-dm-p2-exploracion-a6` se convierte en las piezas que necesita el controlador de la etapa 2: un modelo único de MLflow con los dos riesgos y un adaptador con `fit`, `evaluate`, `input_example` y `plots`. Al final se compara con `mibici_equipo39/modeling/p2.py`. Prueba no se usa.

In [1]:
import tempfile

import mlflow
import numpy as np
import pandas as pd
from mlflow.models import infer_signature

from mibici_equipo39 import config as C
from mibici_equipo39.features import load_p2_targets, load_tables
from mibici_equipo39.modeling.p2 import COLUMNS, FORBIDDEN, TARGETS, choose_threshold, classification_metrics, make_baseline, make_candidate

metadata, X, _ = load_tables()
data = pd.concat([metadata, X, load_p2_targets()], axis=1)
data = data.loc[data["capacity"].notna()].reset_index(drop=True)
data[list(TARGETS.values())] = data[list(TARGETS.values())].astype(int)
print(len(data), "filas; columnas de entrada:", COLUMNS)

4108440 filas; columnas de entrada: ['hour_of_day', 'day_of_week', 'month', 'is_weekend', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'station_id']


## Un solo modelo para los dos riesgos

In [2]:
def positive_proba(model, X):
    positive = np.flatnonzero(model.classes_ == 1)
    if not len(positive):
        return np.zeros(len(X), dtype=float)
    return model.predict_proba(X)[:, positive[0]]


class P2RiskModel(mlflow.pyfunc.PythonModel):
    def __init__(self, pipelines, thresholds):
        self.pipelines = pipelines
        self.thresholds = thresholds

    def predict(self, context, model_input: pd.DataFrame, params=None) -> pd.DataFrame:
        missing = set(COLUMNS) - set(model_input.columns)
        if missing:
            raise ValueError(f"Faltan columnas de entrada: {', '.join(sorted(missing))}.")
        X = model_input[COLUMNS]
        probs = {risk: positive_proba(self.pipelines[risk], X) for risk in TARGETS}
        return pd.DataFrame({**{f"prob_{r}": p for r, p in probs.items()},
                             **{f"alert_{r}": p >= self.thresholds[r] for r, p in probs.items()}},
                            index=model_input.index)

> **Por qué un `PythonModel` y no dos modelos sklearn.** El Model Registry guarda **un modelo por pregunta** (`mibici-equipo39-p2-riesgo`). P2 tiene dos clasificadores y dos umbrales, y los dos se necesitan juntos para decidir una visita. Empaquetarlos en un `PythonModel` hace que quien cargue `models:/mibici-equipo39-p2-riesgo@champion` reciba las cuatro salidas sin tener que conocer los umbrales.
>
> **Por qué `positive_proba` busca la clase 1.** Si en train un riesgo no tuviera positivos, el clasificador solo conoce una clase y `predict_proba` devuelve una sola columna. Buscar la columna de la clase 1 evita leer la columna equivocada o fallar.

## Ajuste con train y calibración con validación

In [3]:
def fit_variant(variant, data):
    train = data.loc[data["split"].eq("train")]
    pipelines = {}
    for risk, target in TARGETS.items():
        if variant["family"] == "dummy" or train[target].nunique() < 2:
            model = make_baseline()
        else:
            model = make_candidate().set_params(**{f"logisticregression__{k}": v for k, v in variant["params"].items()})
        pipelines[risk] = model.fit(train[COLUMNS], train[target])
    return P2RiskModel(pipelines, {})


def calibrate(model, data, frozen=None):
    val = data.loc[data["split"].eq("validation")]
    for risk, target in TARGETS.items():
        score = positive_proba(model.pipelines[risk], val[COLUMNS])
        model.thresholds[risk] = frozen[risk] if frozen else choose_threshold(val[target].to_numpy(), score)[0]
    return model.thresholds

best = next(v for v in C.STAGE2_SEARCH_PLAN["p2"] if v["params"]["C"] == 0.1)
notebook_model = fit_variant(best, data)
calibrate(notebook_model, data)

{'bikes': 0.65, 'docks': 0.7}

> **Por qué la calibración es un paso aparte.** `fit` solo ve train. Los umbrales se eligen después con validación y quedan congelados para prueba; test no participa. Separarlo deja el orden explícito: ajuste con train, calibración con validación, selección de variante y una sola evaluación en test. En una reproducción, `calibrate` recibe los umbrales de `selection.json` y no los vuelve a calcular. Guardarlos dentro del modelo garantiza que el umbral registrado es el que se usa después.

## Evaluación por conjunto

In [4]:
def evaluate(model, data, split):
    prefix = {"train": "train", "validation": "val", "test": "test"}[split]
    frame = data.loc[data["split"].eq(split)]
    output = model.predict(None, frame[COLUMNS])
    metrics = {}
    for risk, target in TARGETS.items():
        y = frame[target].to_numpy()
        values, _ = classification_metrics(y, output[f"prob_{risk}"].to_numpy(), model.thresholds[risk])
        metrics.update({f"{prefix}_{k}_{risk}": v for k, v in values.items()})
        metrics[f"{prefix}_alerts_per_1000_{risk}"] = 1000 * float(output[f"alert_{risk}"].mean())
    metrics[f"{prefix}_macro_f1"] = (metrics[f"{prefix}_f1_bikes"] + metrics[f"{prefix}_f1_docks"]) / 2
    return metrics

pd.Series(evaluate(notebook_model, data, "validation")).round(4)

val_recall_bikes                 0.8089
val_precision_bikes              0.0172
val_f1_bikes                     0.0337
val_support_positive_bikes    2213.0000
val_ap_bikes                     0.0310
val_alerts_per_1000_bikes      151.6874
val_recall_docks                 0.8076
val_precision_docks              0.0264
val_f1_docks                     0.0511
val_support_positive_docks    2926.0000
val_ap_docks                     0.0516
val_alerts_per_1000_docks      130.5373
val_macro_f1                     0.0424
dtype: float64

> **Mismos números que la exploración** para C=0.1 en validación. Las alertas por mil estación-horas se agregan porque muestran la carga de visitas que generaría el modelo.

## Guardar y volver a cargar con MLflow

In [5]:
example = data.loc[data["split"].eq("validation"), COLUMNS].head(3).reset_index(drop=True)
example[COLUMNS[:-1]] = example[COLUMNS[:-1]].astype(float)
example["station_id"] = example["station_id"].astype(str)
signature = infer_signature(example, notebook_model.predict(None, example))
with tempfile.TemporaryDirectory() as folder:
    mlflow.pyfunc.save_model(f"{folder}/p2", python_model=notebook_model, signature=signature, input_example=example)
    loaded = mlflow.pyfunc.load_model(f"{folder}/p2")
    reloaded = loaded.predict(example)
print(signature)
reloaded

2026/10/24 15:26:46 INFO mlflow.pyfunc: Validating input example against model signature


inputs: 
  ['hour_of_day': double (required), 'day_of_week': double (required), 'month': double (required), 'is_weekend': double (required), 'hour_sin': double (required), 'hour_cos': double (required), 'month_sin': double (required), 'month_cos': double (required), 'station_id': string (required)]
outputs: 
  ['prob_bikes': double (required), 'prob_docks': double (required), 'alert_bikes': boolean (required), 'alert_docks': boolean (required)]
params: 
  None



,prob_bikes,prob_docks,alert_bikes,alert_docks
0,0.201362,0.053658,False,False
1,0.213577,0.074933,False,False
2,0.242101,0.111461,False,False


> **Por qué la firma sale de la predicción real.** `infer_signature` toma el ejemplo de entrada y la salida del propio modelo: nueve columnas de entrada (calendario y `station_id`) y cuatro de salida. MLflow la usa para validar a quien llame al modelo registrado. El ejemplo es agregado por estación y hora, sin datos de usuarios.

## Paso al módulo

| Pieza | Destino |
|---|---|
| `positive_proba`, `P2RiskModel` | `mibici_equipo39/modeling/p2.py` |
| `fit_variant`, `calibrate`, `evaluate`, ejemplo y gráficas | métodos de `P2Adapter` en `p2.py`, expuesto como `ADAPTER` |
| Variantes, rejilla, recall mínimo y supuesto de capacidad | `config.py` |
| Matrices, curvas y selección de umbral | `mibici_equipo39/plots.py` |

El controlador lo ejecuta con `mlflow run . -e train -P question=p2`.

In [6]:
from mibici_equipo39.modeling.p2 import ADAPTER

module_fit = ADAPTER.fit(best, data)
ADAPTER.calibrate(module_fit, data)
rows = data.loc[data["split"].eq("validation"), COLUMNS]
assert module_fit.model.thresholds == notebook_model.thresholds
pd.testing.assert_frame_equal(module_fit.model.predict(None, rows), notebook_model.predict(None, rows))
print("notebook y módulo dan los mismos umbrales y salidas:", module_fit.model.thresholds)

notebook y módulo dan los mismos umbrales y salidas: {'bikes': 0.65, 'docks': 0.7}
